# Code mẫu — GAN và diffusion

GAN có hai bước cập nhật, và gradient đi theo hai đường khác nhau ở mỗi bước. Diffusion thì cần dùng đúng một lịch thêm nhiễu cho cả lúc học lẫn lúc sinh. Dữ liệu ở đây chỉ là các số một chiều, để mọi phép toán đều kiểm được trực tiếp. Chúng chưa thay được một model sinh ảnh thật.

## Ví dụ 1: Ở mỗi bước GAN, mạng nào được đổi?

Dữ liệu thật là bốn số quanh 2. Generator, gọi tắt là G, và discriminator, gọi tắt là D, mỗi mạng chỉ là một lớp tuyến tính. Mục tiêu của ví dụ là kiểm gradient đi đâu và optimizer nào chạy, không phải đánh giá chất lượng mẫu sinh hay xem GAN có hội tụ không.

Bước đầu là bước của D: D học phân biệt số thật với số do G sinh ra.

In [ ]:
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(58)
generator = nn.Linear(1, 1)
discriminator = nn.Linear(1, 1)
opt_g = torch.optim.SGD(generator.parameters(), lr=0.1)
opt_d = torch.optim.SGD(discriminator.parameters(), lr=0.1)
criterion = nn.BCEWithLogitsLoss()
real = torch.tensor([[1.7], [1.9], [2.1], [2.3]])
noise = torch.tensor([[-1.], [0.], [1.], [2.]])
g_before = [p.detach().clone() for p in generator.parameters()]
d_before = [p.detach().clone() for p in discriminator.parameters()]
opt_d.zero_grad()
real_logits = discriminator(real)
fake_logits = discriminator(generator(noise).detach())
loss_d = criterion(real_logits, torch.ones_like(real_logits))
loss_d += criterion(fake_logits, torch.zeros_like(fake_logits))
loss_d.backward()
opt_d.step()
print("Bước D, loss:", round(loss_d.item(), 4))
print("G không gradient:", all(p.grad is None for p in generator.parameters()))
print("G giữ nguyên:", all(torch.equal(a, b) for a, b in zip(g_before, generator.parameters())))
print("D đã đổi:", any(not torch.equal(a, b) for a, b in zip(d_before, discriminator.parameters())))
assert all(p.grad is None for p in generator.parameters())

Bước D, loss: 1.4571
G không gradient: True
G giữ nguyên: True
D đã đổi: True


Số do G sinh ra được `detach()` trước khi đưa vào D. Với bước này, chúng chỉ là dữ liệu, giống như số thật. Vì vậy G không nhận gradient nào, và chỉ optimizer của D chạy. Kết quả: D đổi, G đứng yên.

Bước sau là bước của G: G học cách làm D tin rằng số giả là thật. Lần này ta khóa tham số của D, nhưng vẫn phải giữ graph đi qua D. Gradient cần đi từ loss, qua D, về tới G. Nếu bọc lời gọi D trong `no_grad()`, đường đi ấy bị cắt, và G không học được gì.

In [ ]:
d_before = [p.detach().clone() for p in discriminator.parameters()]
g_before = [p.detach().clone() for p in generator.parameters()]
for p in discriminator.parameters():
    p.requires_grad_(False)
opt_g.zero_grad()
fake = generator(noise)
logits = discriminator(fake)
loss_g = criterion(logits, torch.ones_like(logits))
loss_g.backward()
gradient = generator.weight.grad.item()
opt_g.step()
for p in discriminator.parameters():
    p.requires_grad_(True)
print("Bước G, loss:", round(loss_g.item(), 4))
print("Gradient weight G:", round(gradient, 4))
print("D giữ nguyên:", all(torch.equal(a, b) for a, b in zip(d_before, discriminator.parameters())))
print("G đã đổi:", any(not torch.equal(a, b) for a, b in zip(g_before, generator.parameters())))
assert gradient != 0
assert all(torch.equal(a, b) for a, b in zip(d_before, discriminator.parameters()))
assert any(not torch.equal(a, b) for a, b in zip(g_before, generator.parameters()))

Bước G, loss: 1.0656
Gradient weight G: -0.052
D giữ nguyên: True
G đã đổi: True


Gradient của G khác 0, tức loss đã truyền qua D về được G, trong khi D vẫn giữ nguyên. Nhãn trong bước này là 1, “thật”, cho chính các số giả. Đó là cách G được dạy để đánh lừa D. Lưu ý là xong bước, ta mở lại tham số cho D để vòng sau D còn học được.

Hai loss của GAN là hai mục tiêu đối đầu nhau. Không thể chỉ nhìn cả hai cùng giảm mà kết luận model đang sinh tốt.

## Ví dụ 2: Thêm nhiễu, và sai số khi đảo ngược bị phóng đại

Với số sạch $x_0=2$ và nhiễu $\varepsilon=0.5$, ta tính số đã thêm nhiễu $x_t=\sqrt{\bar\alpha}\,x_0+\sqrt{1-\bar\alpha}\,\varepsilon$. Ta so hai mức: $\bar\alpha=0.64$, nhiễu vừa, và $\bar\alpha=0.04$, gần như toàn nhiễu. Ở mỗi mức, ta khôi phục $x_0$ hai lần: một lần với nhiễu đúng, một lần với nhiễu đoán lệch 0.1.

In [ ]:
import numpy as np

clean, noise = 2.0, 0.5
for alpha_bar in [0.64, 0.04]:
    noisy = np.sqrt(alpha_bar) * clean + np.sqrt(1 - alpha_bar) * noise
    exact = (noisy - np.sqrt(1 - alpha_bar) * noise) / np.sqrt(alpha_bar)
    estimated_noise = noise + 0.1
    estimate = (noisy - np.sqrt(1 - alpha_bar) * estimated_noise) / np.sqrt(alpha_bar)
    print("alpha_bar", alpha_bar, "x_t/x0 đúng/x0 ước lượng:",
          [round(float(v), 4) for v in [noisy, exact, estimate]])
    assert np.isclose(exact, clean)

betas = np.array([0.1, 0.2, 0.3])
alpha_bars = np.cumprod(1 - betas)
print("alpha_bar theo lịch:", alpha_bars.round(4).tolist())
assert np.allclose(alpha_bars, [0.9, 0.72, 0.504])
assert np.all(np.diff(alpha_bars) < 0)

alpha_bar 0.64 x_t/x0 đúng/x0 ước lượng: [1.9, 2.0, 1.925]
alpha_bar 0.04 x_t/x0 đúng/x0 ước lượng: [0.8899, 2.0, 1.5101]
alpha_bar theo lịch: [0.9, 0.72, 0.504]


Ở mức 0.64, $x_t=0.8\cdot2+0.6\cdot0.5=1.9$, và biết nhiễu đúng thì khôi phục ra đúng 2. Khi đoán nhiễu lệch 0.1, kết quả chỉ lệch 0.075. Ở mức 0.04, cùng độ lệch 0.1 làm kết quả lệch gần 0.49. Lý do là công thức khôi phục chia cho $\sqrt{\bar\alpha}$, mà $\sqrt{0.04}$ chỉ là 0.2. Càng nhiều nhiễu, một lỗi nhỏ khi đoán nhiễu càng bị phóng đại.

Ba dòng cuối tính $\bar\alpha$ từ lịch $\beta$. Mỗi $\bar\alpha_t$ là tích của mọi $1-\beta$ từ đầu tới bước t, chứ không phải chỉ $1-\beta_t$ của riêng bước đó. Vì vậy $\bar\alpha$ giảm dần theo thời gian.

## Ví dụ 3: Học dự đoán nhiễu, rồi đi ngược theo cùng lịch

Dữ liệu sạch là các số nằm quanh −2 hoặc quanh +2. Một MLP nhận vào số đã thêm nhiễu và thời điểm t, đã chia về khoảng 0 tới 1. Đích của nó là đoán ra nhiễu đã được thêm. Validation được sinh một lần rồi giữ cố định. Ta so MSE của model với một baseline luôn đoán nhiễu bằng 0.

In [ ]:
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(580)
steps = 40
betas = torch.linspace(0.02, 0.18, steps)
alpha_bars = torch.cumprod(1 - betas, dim=0)
denoiser = nn.Sequential(nn.Linear(2, 24), nn.Tanh(), nn.Linear(24, 1))
optimizer = torch.optim.Adam(denoiser.parameters(), lr=0.01)

def batch(count):
    clean = (torch.randint(0, 2, (count, 1)) * 4 - 2).float() + 0.2 * torch.randn(count, 1)
    times = torch.randint(0, steps, (count,))
    noise = torch.randn(count, 1)
    a = alpha_bars[times, None]
    noisy = a.sqrt() * clean + (1 - a).sqrt() * noise
    return torch.cat([noisy, times[:, None] / (steps - 1)], dim=1), noise

valid_x, valid_noise = batch(128)
with torch.no_grad():
    before = nn.functional.mse_loss(denoiser(valid_x), valid_noise).item()
for update in range(180):
    x, target = batch(64)
    optimizer.zero_grad()
    loss = nn.functional.mse_loss(denoiser(x), target)
    loss.backward()
    optimizer.step()
denoiser.eval()
with torch.no_grad():
    after = nn.functional.mse_loss(denoiser(valid_x), valid_noise).item()
    baseline = valid_noise.square().mean().item()
print("Noise MSE trước/sau/baseline 0:", [round(v, 4) for v in [before, after, baseline]])
print("alpha_bar cuối:", round(alpha_bars[-1].item(), 4))
assert after < before and after < baseline

Noise MSE trước/sau/baseline 0: [0.8922, 0.3979, 0.9815]
alpha_bar cuối: 0.014


Sau 180 bước, MSE giảm từ 0.89 xuống 0.40, thấp hơn hẳn baseline 0.98. Model đã học được phần nào cách đoán nhiễu trên loại dữ liệu này. $\bar\alpha$ ở bước cuối khoảng 0.014, tức tín hiệu sạch chỉ còn rất ít. Vì vậy khi sinh, ta có thể bắt đầu từ nhiễu chuẩn thuần. Dù vậy, với số bước hữu hạn, tín hiệu sạch chưa bao giờ về đúng 0.

Giờ đi ngược để sinh mẫu. Ta dùng một cách cập nhật không ngẫu nhiên, theo kiểu DDIM với `eta = 0`. Ở mỗi bước, model đoán nhiễu, ta ước lượng số sạch từ đó, rồi dựng lại số ở mức nhiễu nhỏ hơn bằng chính nhiễu vừa đoán. Đây là bộ sinh một chiều để minh họa, không phải DDPM có thêm nhiễu ngẫu nhiên mỗi bước.

In [ ]:
torch.manual_seed(581)
samples = torch.randn(8, 1)
with torch.no_grad():
    for t in reversed(range(steps)):
        current = alpha_bars[t]
        previous = alpha_bars[t - 1] if t > 0 else torch.tensor(1.0)
        time_feature = torch.full_like(samples, t / (steps - 1))
        predicted_noise = denoiser(torch.cat([samples, time_feature], dim=1))
        estimated_clean = (samples - (1 - current).sqrt() * predicted_noise) / current.sqrt()
        samples = previous.sqrt() * estimated_clean + (1 - previous).sqrt() * predicted_noise
print("Tám mẫu sinh một chiều:", [round(v, 3) for v in samples.flatten().tolist()])
print("Mẫu âm/dương:", int((samples < 0).sum()), int((samples >= 0).sum()))
assert torch.isfinite(samples).all()

Tám mẫu sinh một chiều: [3.317, 0.837, 2.378, -1.349, -1.826, 1.957, 0.837, -1.85]
Mẫu âm/dương: 3 5


Các mẫu sinh ra có cả âm lẫn dương, tức model đã tìm về cả hai cụm. Nhưng 0.837 và 3.317 còn khá xa cụm +2. MSE đoán nhiễu tốt hơn baseline chưa có nghĩa là các mẫu sinh ra đã giống dữ liệu thật. Tám mẫu cũng quá ít để biết hai cụm có được phủ đều không.

Mọi hệ số trong vòng đi ngược đều lấy đúng từ `alpha_bars` của lịch đã dùng khi train. Dùng một lịch khác lúc sinh là một lỗi khó phát hiện. Ở bước cuối, `previous` bằng 1, nên phần nhiễu biến mất, và mẫu trả ra chính là ước lượng số sạch.